# Fenway Park Game & Weather Dataset (2016-2025)

This notebook builds a comprehensive dataset where each row represents a regular season game played at Fenway Park. It combines:
1. **Game statistics** from Statcast pitch-by-pitch data (pybaseball)
2. **Weather data** from Meteostat hourly observations
3. **Wind projections** onto outfield vectors (CF, LCF, RCF)

All batting/pitching statistics are **both teams combined** to capture the full park-environment effect.

## Section 0: Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import time
import os
import requests
import warnings
warnings.filterwarnings('ignore')

# pybaseball
from pybaseball import statcast
from pybaseball import cache
cache.enable()  # Cache Statcast data locally to avoid re-downloading

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

SEASONS = range(2016, 2026)  # 2016 through 2025
print("Setup complete. Seasons:", list(SEASONS))

## Section 1: Pull Statcast Pitch-Level Data

Pull pitch-by-pitch Statcast data year-by-year for all Red Sox games, then filter to regular season home games at Fenway.

In [ ]:
# Pull Statcast data month-by-month to avoid parser errors from large requests
# Baseball Savant can return malformed data on big date ranges; smaller chunks are more reliable

all_pitches = []

MONTH_RANGES = [
    ('03-01', '03-31'), ('04-01', '04-30'), ('05-01', '05-31'),
    ('06-01', '06-30'), ('07-01', '07-31'), ('08-01', '08-31'),
    ('09-01', '09-30'), ('10-01', '10-31'), ('11-01', '11-30'),
]

for year in SEASONS:
    year_pitches = 0
    print(f"Pulling {year} season...")
    
    for m_start, m_end in MONTH_RANGES:
        start = f"{year}-{m_start}"
        end = f"{year}-{m_end}"
        
        for attempt in range(3):  # Up to 3 retries per month
            try:
                df = statcast(start_dt=start, end_dt=end, team='BOS', verbose=False)
                if df is not None and len(df) > 0:
                    all_pitches.append(df)
                    year_pitches += len(df)
                break  # Success — exit retry loop
            except Exception as e:
                if attempt < 2:
                    print(f"  Retry {attempt+1} for {start} to {end}: {e}")
                    time.sleep(2)
                else:
                    print(f"  FAILED {start} to {end} after 3 attempts: {e}")
    
    print(f"  {year}: {year_pitches:,} pitches")

pitches_raw = pd.concat(all_pitches, ignore_index=True)
print(f"\nTotal raw pitches (home + away): {len(pitches_raw):,}")

In [ ]:
# Filter to regular season home games at Fenway only
pitches = pitches_raw[
    (pitches_raw['game_type'] == 'R') &
    (pitches_raw['home_team'] == 'BOS')
].copy()

pitches['game_date'] = pd.to_datetime(pitches['game_date'])
pitches['season'] = pitches['game_date'].dt.year

print(f"Filtered pitches (regular season, Fenway): {len(pitches):,}")
print(f"Unique games: {pitches['game_pk'].nunique()}")
print(f"Seasons: {sorted(pitches['season'].unique())}")
print(f"\nGames per season:")
print(pitches.groupby('season')['game_pk'].nunique())

## Section 2: Aggregate Pitch Data to Game Level

Compute game-level statistics from pitch-by-pitch data. All stats are **both teams combined**.

In [ ]:
# Define event categories
HIT_EVENTS = {'single', 'double', 'triple', 'home_run'}
BATTED_BALL_EVENTS = HIT_EVENTS | {
    'field_out', 'grounded_into_double_play', 'force_out',
    'fielders_choice_out', 'sac_fly', 'field_error',
    'double_play', 'sac_bunt', 'fielders_choice',
    'sac_fly_double_play', 'triple_play'
}

def is_barrel(ev, la):
    """
    MLB barrel definition:
    - Minimum 98 mph exit velocity
    - At 98 mph: launch angle 26-30 degrees
    - For each mph above 98, the LA range expands
    - At 116+ mph: launch angle 8-50 degrees
    """
    if pd.isna(ev) or pd.isna(la):
        return False
    if ev < 98:
        return False
    excess_mph = min(ev - 98, 18)
    la_min = 26 - (excess_mph * 1.0)
    la_max = 30 + (excess_mph * 1.111)
    return la_min <= la <= la_max


def aggregate_game(game_df):
    """Aggregate pitch-level data to a single game row."""
    game_pk = game_df['game_pk'].iloc[0]
    game_date = game_df['game_date'].iloc[0]
    season = game_df['season'].iloc[0]
    away_team = game_df['away_team'].iloc[0]

    # --- RUNS: final score from last pitch of the game ---
    last_pitch = game_df.sort_values(
        ['inning', 'at_bat_number', 'pitch_number'], ascending=True
    ).iloc[-1]
    home_runs_scored = last_pitch['post_home_score'] if pd.notna(last_pitch.get('post_home_score')) else np.nan
    away_runs_scored = last_pitch['post_away_score'] if pd.notna(last_pitch.get('post_away_score')) else np.nan
    total_runs = home_runs_scored + away_runs_scored if pd.notna(home_runs_scored) and pd.notna(away_runs_scored) else np.nan

    # --- EVENTS (plate appearance results only) ---
    pa_df = game_df[game_df['events'].notna()]
    home_runs_hit = (pa_df['events'] == 'home_run').sum()
    strikeouts = pa_df['events'].isin(['strikeout', 'strikeout_double_play']).sum()
    walks = pa_df['events'].isin(['walk', 'intent_walk']).sum()
    hits = pa_df['events'].isin(HIT_EVENTS).sum()

    # --- TOTAL PITCHES ---
    total_pitches = len(game_df)

    # --- BATTED BALL METRICS ---
    batted = game_df[game_df['launch_speed'].notna()]
    avg_exit_velocity = batted['launch_speed'].mean() if len(batted) > 0 else np.nan

    # Barrel rate: barrels / batted ball events
    bbe_df = pa_df[pa_df['events'].isin(BATTED_BALL_EVENTS)]
    n_bbe = len(bbe_df)
    if n_bbe > 0:
        n_barrels = sum(is_barrel(row['launch_speed'], row['launch_angle']) for _, row in bbe_df.iterrows())
        barrel_rate = n_barrels / n_bbe
    else:
        n_barrels = 0
        barrel_rate = np.nan

    # HR / H ratio
    hr_h_ratio = home_runs_hit / hits if hits > 0 else np.nan

    return pd.Series({
        'game_pk': game_pk,
        'game_date': game_date,
        'season': season,
        'away_team': away_team,
        'home_runs_scored': home_runs_scored,
        'away_runs_scored': away_runs_scored,
        'total_runs': total_runs,
        'home_runs_hit': home_runs_hit,
        'strikeouts': strikeouts,
        'walks': walks,
        'hits': hits,
        'total_pitches': total_pitches,
        'avg_exit_velocity': avg_exit_velocity,
        'n_barrels': n_barrels,
        'n_bbe': n_bbe,
        'barrel_rate': barrel_rate,
        'hr_h_ratio': hr_h_ratio,
    })


print("Aggregating pitch data to game level...")
games = pitches.groupby('game_pk').apply(aggregate_game).reset_index(drop=True)
games['game_date'] = pd.to_datetime(games['game_date'])

# Ensure integer types for count columns
int_cols = ['game_pk', 'home_runs_scored', 'away_runs_scored', 'total_runs',
            'home_runs_hit', 'strikeouts', 'walks', 'hits', 'total_pitches',
            'n_barrels', 'n_bbe', 'season']
for col in int_cols:
    games[col] = pd.to_numeric(games[col], errors='coerce').astype('Int64')

games = games.sort_values('game_date').reset_index(drop=True)

print(f"Total games: {len(games)}")
print(f"\nGames per season:")
print(games.groupby('season')['game_pk'].count())
print(f"\nSample:")
print(games.head(3))

## Section 3: Get Game Start Times

Statcast data does not include game start times. Use the MLB Stats API schedule endpoint to get actual start times (in UTC), convert to Eastern, and merge on `game_pk`.

In [ ]:
# Pull game start times from MLB Stats API (teamId=111 = Red Sox)
MLB_API_URL = "https://statsapi.mlb.com/api/v1/schedule"
RED_SOX_ID = 111

all_start_times = []

for year in SEASONS:
    print(f"Pulling {year} schedule from MLB Stats API...")
    resp = requests.get(MLB_API_URL, params={
        'teamId': RED_SOX_ID,
        'season': year,
        'sportId': 1,
        'gameType': 'R',
    })
    resp.raise_for_status()
    data = resp.json()
    
    for date_entry in data.get('dates', []):
        for game in date_entry.get('games', []):
            home_team_id = game.get('teams', {}).get('home', {}).get('team', {}).get('id')
            if home_team_id == RED_SOX_ID:
                all_start_times.append({
                    'game_pk': game['gamePk'],
                    'game_datetime_utc': game['gameDate'],
                })

start_times_df = pd.DataFrame(all_start_times)

# Convert UTC ISO timestamps to Eastern time
start_times_df['game_start'] = (
    pd.to_datetime(start_times_df['game_datetime_utc'])
    .dt.tz_convert('America/New_York')
    .dt.tz_localize(None)  # Remove timezone info for clean merging
)
start_times_df['start_hour'] = start_times_df['game_start'].dt.hour
start_times_df['game_pk'] = start_times_df['game_pk'].astype('Int64')

print(f"\nHome game start times retrieved: {len(start_times_df)}")
print(start_times_df[['game_pk', 'game_start', 'start_hour']].head())

In [ ]:
# Merge start times into game data on game_pk (unique game ID — handles doubleheaders naturally)
games = games.merge(
    start_times_df[['game_pk', 'game_start', 'start_hour']],
    on='game_pk',
    how='left'
)

n_missing = games['game_start'].isna().sum()
print(f"Games with start times: {len(games) - n_missing} / {len(games)}")
if n_missing > 0:
    print(f"\nWARNING: {n_missing} games missing start times:")
    print(games[games['game_start'].isna()][['game_pk', 'game_date', 'season']])

## Section 4: Pull Weather Data (Meteostat)

Use Meteostat to pull hourly weather data for each season, then average over the 3 hours following each game's start time.

In [31]:
# Fenway Park coordinates
FENWAY_LAT = 42.346268
FENWAY_LON = -71.095764

# Using Open-Meteo Historical Weather API (free, no key required)
# Returns ERA5 reanalysis + station data for any lat/lon
OPEN_METEO_URL = "https://archive-api.open-meteo.com/v1/archive"
HOURLY_PARAMS = "temperature_2m,relative_humidity_2m,surface_pressure,precipitation,wind_speed_10m,wind_direction_10m"

# Test fetch to confirm API is reachable
test_resp = requests.get(OPEN_METEO_URL, params={
    'latitude': FENWAY_LAT,
    'longitude': FENWAY_LON,
    'start_date': '2023-07-01',
    'end_date': '2023-07-02',
    'hourly': HOURLY_PARAMS,
    'timezone': 'America/New_York',
})

if test_resp.status_code == 200:
    test_data = test_resp.json()
    n_hours = len(test_data['hourly']['time'])
    print(f"Open-Meteo API test (Jul 1-2 2023): {n_hours} hourly records - OK")
    print(f"Sample time: {test_data['hourly']['time'][12]}")
    print(f"Sample temp: {test_data['hourly']['temperature_2m'][12]}°C")
    print(f"Sample wind: {test_data['hourly']['wind_speed_10m'][12]} km/h from {test_data['hourly']['wind_direction_10m'][12]}°")
else:
    print(f"ERROR: Open-Meteo API returned {test_resp.status_code}")
    print(test_resp.text)

Open-Meteo API test (Jul 1-2 2023): 48 hourly records - OK
Sample time: 2023-07-01T12:00
Sample temp: 23.5°C
Sample wind: 13.9 km/h from 80°


In [32]:
def fetch_season_weather(year):
    """Fetch hourly weather for a full season from Open-Meteo."""
    resp = requests.get(OPEN_METEO_URL, params={
        'latitude': FENWAY_LAT,
        'longitude': FENWAY_LON,
        'start_date': f'{year}-03-01',
        'end_date': f'{year}-11-30',
        'hourly': HOURLY_PARAMS,
        'timezone': 'America/New_York',  # Returns Eastern time directly
    })
    resp.raise_for_status()
    hourly = resp.json()['hourly']
    
    df = pd.DataFrame({
        'temp': hourly['temperature_2m'],
        'rhum': hourly['relative_humidity_2m'],
        'pres': hourly['surface_pressure'],
        'prcp': hourly['precipitation'],
        'wspd': hourly['wind_speed_10m'],
        'wdir': hourly['wind_direction_10m'],
    }, index=pd.to_datetime(hourly['time']))
    
    return df


def get_game_weather(game_start_dt, hourly_df):
    """
    Average weather over the 3 hours following game start.
    game_start_dt: datetime (local Boston time, rounded to hour)
    hourly_df: DataFrame with hourly weather, index is naive Eastern time
    """
    start = game_start_dt
    end = start + timedelta(hours=2)  # 3 hourly obs: start, +1h, +2h
    
    window = hourly_df.loc[start:end]
    
    if len(window) == 0:
        return pd.Series({
            'temp_c': np.nan, 'rhum': np.nan, 'pres': np.nan,
            'prcp': np.nan, 'wspd': np.nan, 'wdir': np.nan
        })
    
    result = {
        'temp_c': window['temp'].mean(),
        'rhum': window['rhum'].mean(),
        'pres': window['pres'].mean(),
        'prcp': window['prcp'].sum(),   # Precipitation SUMMED (cumulative quantity)
        'wspd': window['wspd'].mean(),
    }
    
    # Wind direction: circular mean to handle 0/360 boundary
    wdir_vals = window['wdir'].dropna()
    if len(wdir_vals) > 0:
        wdir_rad = np.radians(wdir_vals)
        mean_sin = np.sin(wdir_rad).mean()
        mean_cos = np.cos(wdir_rad).mean()
        result['wdir'] = np.degrees(np.arctan2(mean_sin, mean_cos)) % 360
    else:
        result['wdir'] = np.nan
    
    return pd.Series(result)


# Pull weather season by season
weather_records = []

for year in SEASONS:
    print(f"Pulling weather for {year}...")
    
    try:
        hourly_df = fetch_season_weather(year)
    except Exception as e:
        print(f"  WARNING: Failed for {year}: {e}")
        season_games = games[games['season'] == year]
        for idx, game in season_games.iterrows():
            weather_records.append({
                'game_pk': game['game_pk'],
                'temp_c': np.nan, 'rhum': np.nan, 'pres': np.nan,
                'prcp': np.nan, 'wspd': np.nan, 'wdir': np.nan
            })
        continue
    
    print(f"  {year}: {len(hourly_df)} hourly records")
    
    season_games = games[games['season'] == year]
    for idx, game in season_games.iterrows():
        if pd.isna(game['game_start']):
            weather_records.append({
                'game_pk': game['game_pk'],
                'temp_c': np.nan, 'rhum': np.nan, 'pres': np.nan,
                'prcp': np.nan, 'wspd': np.nan, 'wdir': np.nan
            })
            continue
        
        game_hour = game['game_start'].replace(minute=0, second=0, microsecond=0)
        wx = get_game_weather(game_hour, hourly_df)
        wx['game_pk'] = game['game_pk']
        weather_records.append(wx.to_dict())

weather_df = pd.DataFrame(weather_records)
weather_df['game_pk'] = weather_df['game_pk'].astype('Int64')
print(f"\nWeather records: {len(weather_df)}")
print(f"Missing temp data: {weather_df['temp_c'].isna().sum()}")
print(weather_df.head(3))

Pulling weather for 2016...
  2016: 6600 hourly records
Pulling weather for 2017...
  2017: 6600 hourly records
Pulling weather for 2018...
  2018: 6600 hourly records
Pulling weather for 2019...
  2019: 6600 hourly records
Pulling weather for 2020...
  2020: 6600 hourly records
Pulling weather for 2021...
  2021: 6600 hourly records
Pulling weather for 2022...
  2022: 6600 hourly records
Pulling weather for 2023...
  2023: 6600 hourly records
Pulling weather for 2024...
  2024: 6600 hourly records
Pulling weather for 2025...
  2025: 6600 hourly records

Weather records: 783
Missing temp data: 0
      temp_c       rhum         pres  prcp       wspd        wdir  game_pk
0  13.300000  52.000000  1018.033333   0.0  27.066667  196.333330   446958
1   6.700000  72.666667  1021.166667   0.0  16.933333  311.338055   446970
2   4.466667  75.000000  1026.933333   0.0   7.366667  129.000000   446983


## Section 5: Wind Direction Bucketing & Outfield Projections

**Wind direction bucketing**: 8 compass directions (N, NE, E, SE, S, SW, W, NW).

**Wind projections**: Project wind onto vectors from home plate to center field (CF), left-center field (LCF), and right-center field (RCF). Positive = blowing out, negative = blowing in.

Fenway outfield directions (degrees from north):
- Center field: ~45° (NE)
- Left-center field: ~25° (NNE)
- Right-center field: ~65° (ENE)

**Important**: Weather APIs report wind direction as the direction wind blows **FROM**. We must convert to the direction it blows **TO** before projecting.

In [ ]:
# Merge weather into game data
games_full = games.merge(weather_df, on='game_pk', how='left')

# --- Wind direction bucketing ---
def bucket_wind_dir(deg):
    """Bucket wind direction (degrees) into 8 compass directions."""
    if pd.isna(deg):
        return np.nan
    buckets = ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW']
    idx = int(((deg + 22.5) % 360) / 45)
    return buckets[idx]

games_full['wind_dir_bucket'] = games_full['wdir'].apply(bucket_wind_dir)

# --- Wind projections onto outfield vectors ---
CF_DIR = 45.0    # Center field: NE
LCF_DIR = 25.0   # Left-center field: NNE
RCF_DIR = 65.0   # Right-center field: ENE

def compute_wind_projection(wdir, wspd, outfield_dir):
    """
    Project wind onto an outfield direction vector.
    
    wdir: direction wind blows FROM (meteostat convention, degrees)
    wspd: wind speed (km/h)
    outfield_dir: compass bearing from home plate to outfield (degrees from north)
    
    Returns: positive = blowing OUT toward outfield, negative = blowing IN
    """
    if pd.isna(wdir) or pd.isna(wspd):
        return np.nan
    # Wind blows FROM wdir, so it travels TOWARD (wdir + 180)
    wind_toward = (wdir + 180) % 360
    # Project onto outfield direction
    angle_diff = wind_toward - outfield_dir
    return wspd * np.cos(np.radians(angle_diff))

games_full['wind_cf'] = games_full.apply(
    lambda r: compute_wind_projection(r['wdir'], r['wspd'], CF_DIR), axis=1
)
games_full['wind_lcf'] = games_full.apply(
    lambda r: compute_wind_projection(r['wdir'], r['wspd'], LCF_DIR), axis=1
)
games_full['wind_rcf'] = games_full.apply(
    lambda r: compute_wind_projection(r['wdir'], r['wspd'], RCF_DIR), axis=1
)

print("Wind projection summary (positive = blowing out, negative = blowing in):")
print(games_full[['wind_cf', 'wind_lcf', 'wind_rcf']].describe())

## Section 6: Final Assembly

Convert units, order columns, and round to sensible precision.

In [34]:
# Unit conversions
games_full['temp_f'] = games_full['temp_c'] * 9/5 + 32
games_full['wspd_mph'] = games_full['wspd'] * 0.621371

# Final column order
final_columns = [
    # Game identification
    'game_pk', 'game_date', 'season', 'away_team', 'game_start', 'start_hour',
    # Scoring
    'home_runs_scored', 'away_runs_scored', 'total_runs',
    # Batting stats (both teams combined)
    'home_runs_hit', 'strikeouts', 'walks', 'hits',
    'total_pitches', 'avg_exit_velocity',
    'n_barrels', 'n_bbe', 'barrel_rate', 'hr_h_ratio',
    # Weather
    'temp_f', 'temp_c', 'rhum', 'pres', 'prcp',
    'wspd', 'wspd_mph', 'wdir', 'wind_dir_bucket',
    # Wind projections
    'wind_cf', 'wind_lcf', 'wind_rcf',
]

fenway_data = games_full[final_columns].copy()
fenway_data = fenway_data.sort_values('game_date').reset_index(drop=True)

# Round floating point columns
round_map = {
    'avg_exit_velocity': 1, 'barrel_rate': 4, 'hr_h_ratio': 4,
    'temp_f': 1, 'temp_c': 1, 'rhum': 1, 'pres': 1, 'prcp': 2,
    'wspd': 1, 'wspd_mph': 1, 'wdir': 1,
    'wind_cf': 2, 'wind_lcf': 2, 'wind_rcf': 2,
}
for col, decimals in round_map.items():
    fenway_data[col] = fenway_data[col].round(decimals)

print(f"Final dataset: {fenway_data.shape[0]} rows x {fenway_data.shape[1]} columns")

Final dataset: 831 rows x 31 columns


## Section 7: Validation

Verify row counts per season, check for nulls, and sanity-check summary statistics.

In [35]:
print("=" * 70)
print("VALIDATION REPORT")
print("=" * 70)

# 1. Row counts per season
print("\n--- Games per Season ---")
season_counts = fenway_data.groupby('season').size()
for year, count in season_counts.items():
    if year == 2020:
        expected = (25, 35)  # COVID shortened season
    else:
        expected = (75, 85)  # Normal: ~81 home games
    status = "OK" if expected[0] <= count <= expected[1] else "WARNING"
    print(f"  {year}: {count} games [{status}] (expected {expected[0]}-{expected[1]})")
print(f"  TOTAL: {len(fenway_data)} games")

# 2. Null check
print("\n--- Null Counts ---")
key_cols = ['total_runs', 'home_runs_hit', 'strikeouts', 'walks',
            'total_pitches', 'avg_exit_velocity', 'barrel_rate',
            'temp_f', 'wspd', 'wdir', 'wind_cf', 'game_start']
for col in key_cols:
    n_null = fenway_data[col].isna().sum()
    pct = 100 * n_null / len(fenway_data)
    status = "OK" if pct < 5 else "WARNING"
    print(f"  {col}: {n_null} nulls ({pct:.1f}%) [{status}]")

# 3. Summary statistics sanity checks
print("\n--- Sanity Checks ---")
checks = [
    ('Avg total runs/game', fenway_data['total_runs'].mean(), '~8-10'),
    ('Avg HR/game', fenway_data['home_runs_hit'].mean(), '~2-3'),
    ('Avg K/game', fenway_data['strikeouts'].mean(), '~16-18'),
    ('Avg BB/game', fenway_data['walks'].mean(), '~6-7'),
    ('Avg exit velocity', fenway_data['avg_exit_velocity'].mean(), '~87-89 mph'),
    ('Avg barrel rate', fenway_data['barrel_rate'].mean(), '~0.06-0.08'),
    ('Avg game temp', fenway_data['temp_f'].mean(), '~60-70 F'),
    ('Min game temp', fenway_data['temp_f'].min(), '>30 F'),
    ('Max game temp', fenway_data['temp_f'].max(), '<105 F'),
    ('Avg wind speed (km/h)', fenway_data['wspd'].mean(), '~10-20 km/h'),
]
for label, val, expected in checks:
    print(f"  {label}: {val:.2f} (expected {expected})")

# 4. Full summary statistics
print("\n--- Summary Statistics ---")
print(fenway_data.describe().T[['mean', 'std', 'min', 'max']].to_string())

VALIDATION REPORT

--- Games per Season ---
  2016: 81 games [OK] (expected 75-85)
  2017: 90 games [WARNING] (expected 75-85)
  2018: 87 games [WARNING] (expected 75-85)
  2019: 90 games [WARNING] (expected 75-85)
  2020: 30 games [OK] (expected 25-35)
  2021: 96 games [WARNING] (expected 75-85)
  2022: 81 games [OK] (expected 75-85)
  2023: 96 games [WARNING] (expected 75-85)
  2024: 87 games [WARNING] (expected 75-85)
  2025: 93 games [WARNING] (expected 75-85)
  TOTAL: 831 games

--- Null Counts ---
  total_runs: 0 nulls (0.0%) [OK]
  home_runs_hit: 0 nulls (0.0%) [OK]
  strikeouts: 0 nulls (0.0%) [OK]
  walks: 0 nulls (0.0%) [OK]
  total_pitches: 0 nulls (0.0%) [OK]
  avg_exit_velocity: 0 nulls (0.0%) [OK]
  barrel_rate: 0 nulls (0.0%) [OK]
  temp_f: 0 nulls (0.0%) [OK]
  wspd: 0 nulls (0.0%) [OK]
  wdir: 0 nulls (0.0%) [OK]
  wind_cf: 0 nulls (0.0%) [OK]
  game_start: 0 nulls (0.0%) [OK]

--- Sanity Checks ---
  Avg total runs/game: 9.66 (expected ~8-10)
  Avg HR/game: 1.11 (expe

In [36]:
# Print dataset header for inspection
print("\n--- First 10 Rows ---")
fenway_data.head(10)


--- First 10 Rows ---


,game_pk,game_date,season,away_team,game_start,start_hour,home_runs_scored,away_runs_scored,total_runs,home_runs_hit,strikeouts,walks,hits,total_pitches,avg_exit_velocity,n_barrels,n_bbe,barrel_rate,hr_h_ratio,temp_f,temp_c,rhum,pres,prcp,wspd,wspd_mph,wdir,wind_dir_bucket,wind_cf,wind_lcf,wind_rcf
0,446958,2016-04-11,2016,BAL,2016-04-11 14:05:00,14,6,9,15,2,15,4,9,158,84.4,4,20,0.2000,0.2222,55.9,13.3,52.0,1018.0,0.0,27.1,16.8,196.3,S,16.97,23.16,8.74
1,446970,2016-04-12,2016,BAL,2016-04-12 19:10:00,19,5,9,14,3,9,5,11,174,82.9,3,28,0.1071,0.2727,44.1,6.7,72.7,1021.2,0.0,16.9,10.5,311.3,NW,7.47,1.82,12.21
2,446983,2016-04-13,2016,BAL,2016-04-13 19:10:00,19,4,2,6,1,10,5,9,167,85.7,3,24,0.1250,0.1111,40.0,4.5,75.0,1026.9,0.0,7.4,4.6,129.0,SE,-3.52,-1.09,-5.52
3,447020,2016-04-15,2016,TOR,2016-04-15 19:10:00,19,5,3,8,2,12,2,3,146,82.9,4,17,0.2353,0.6667,44.2,6.8,59.0,1028.9,0.0,12.8,7.9,47.0,NE,-11.96,-12.77,-9.71
4,447035,2016-04-16,2016,TOR,2016-04-16 16:05:00,16,4,2,6,0,12,0,7,135,81.5,1,20,0.0500,0.0000,47.4,8.5,58.3,1030.5,0.0,14.0,8.7,68.7,E,-14.03,-13.09,-13.28
5,447050,2016-04-17,2016,TOR,2016-04-17 13:35:00,13,1,5,6,1,12,1,14,176,83.7,0,28,0.0000,0.0714,58.5,14.7,42.3,1026.5,0.0,8.5,5.3,77.7,E,-8.33,-7.32,-8.34
6,447057,2016-04-18,2016,TOR,2016-04-18 11:05:00,11,1,4,5,0,5,7,8,166,83.2,0,25,0.0000,0.0000,60.3,15.7,51.7,1022.1,0.0,15.8,9.8,61.7,NE,-15.72,-15.32,-14.22
7,447071,2016-04-19,2016,TB,2016-04-19 19:10:00,19,0,3,3,1,13,5,6,179,81.0,2,24,0.0833,0.1667,49.0,9.4,47.7,1016.4,0.0,5.4,3.4,355.3,N,-1.65,-3.31,0.21
8,447086,2016-04-20,2016,TB,2016-04-20 19:10:00,19,7,3,10,1,11,2,7,138,84.8,3,23,0.1304,0.1429,45.6,7.5,69.7,1022.2,0.0,15.5,9.6,191.0,S,8.56,12.46,3.63
9,447100,2016-04-21,2016,TB,2016-04-21 13:35:00,13,8,12,20,2,8,6,15,186,81.9,3,33,0.0909,0.1333,70.4,21.3,30.3,1013.7,0.0,14.0,8.7,227.3,SW,13.11,13.97,10.67


## Section 8: Save to CSV

In [37]:
# Save final dataset
output_dir = os.path.join('..', 'Final Datasets')
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, 'fenway_data_2016.csv')
fenway_data.to_csv(output_path, index=False)

print(f"Saved to: {os.path.abspath(output_path)}")
print(f"File size: {os.path.getsize(output_path) / 1024:.1f} KB")
print(f"Rows: {len(fenway_data)}, Columns: {len(fenway_data.columns)}")

# Verify roundtrip
verify = pd.read_csv(output_path)
assert verify.shape == fenway_data.shape, f"Shape mismatch: {verify.shape} vs {fenway_data.shape}"
print("\nSave & reload verification: PASSED")

Saved to: /Users/avabrown/Desktop/DATASCI 192A/Stadium Datasets/Final Datasets/fenway_data_2016.csv
File size: 122.8 KB
Rows: 831, Columns: 31

Save & reload verification: PASSED
